In [ ]:
#| default_exp train_flow

# Flow Matching Generative Model

Trains a flow matching model on pre-encoded (optionally PCA-reduced) embeddings.
Source distribution is N(0,I); target is the embedding distribution.
Uses RK4 integration and optional time warping at inference.

In [ ]:
#| export
import gc
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np
import wandb
from midi_rae.data import EmbeddingDataset
from midi_rae.utils import *


In [ ]:
#| export
class VelocityNet(nn.Module):
    """MLP velocity field for flow matching.  Input: [x, (x_self_cond,) t_emb], output: dx/dt.
    Hidden layers use residual (skip) connections.
    t_dim: sinusoidal time embedding dim (replaces bare scalar t).
    self_condition: if True, also accepts x_self_cond (predicted x1 from prior pass); zeros when absent."""
    def __init__(self, input_dim, h_dim=256, n_layers=3, self_condition=False, t_dim=64):
        super().__init__()
        self.self_condition = self_condition
        self.t_dim = t_dim
        net_in = input_dim * 2 + t_dim if self_condition else input_dim + t_dim
        self.fc_in  = nn.Linear(net_in, h_dim)
        self.hidden = nn.ModuleList([nn.Linear(h_dim, h_dim) for _ in range(n_layers - 1)])
        self.fc_out = nn.Linear(h_dim, input_dim)

    def forward(self, x, t, x_self_cond=None):
        if t.dim() > 1: t = t.squeeze(-1)
        elif t.dim() == 0: t = t.unsqueeze(0).expand(x.size(0))
        t_emb = sinusoidal_time_emb(t, self.t_dim)          # [B, t_dim]
        if self.self_condition:
            sc = x_self_cond if x_self_cond is not None else torch.zeros_like(x)
            inp = torch.cat([x, sc, t_emb], dim=1)
        else:
            inp = torch.cat([x, t_emb], dim=1)
        h = F.gelu(self.fc_in(inp))
        for layer in self.hidden:
            h = F.gelu(layer(h)) + h
        return self.fc_out(h)


In [ ]:
#| export
import math

def sinusoidal_time_emb(t, dim=64):
    """Sinusoidal time embedding (à la DDPM/DiT).  t: [B] or [B,1] → [B, dim].
    Gives the model a rich multi-frequency view of t instead of a bare scalar."""
    if t.dim() > 1: t = t.squeeze(-1)
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, dtype=torch.float32, device=t.device) / (half - 1))
    x = t.float().unsqueeze(1) * freqs.unsqueeze(0)   # [B, half]
    return torch.cat([x.sin(), x.cos()], dim=-1)       # [B, dim]


In [ ]:
#| export
class PerLevelFlowModel(nn.Module):
    """One VelocityNet per embedding level; each level's slice is routed to its own net.
    Has the same forward(x, t, x_self_cond=None) interface as VelocityNet.
    level_dims: list of ints, e.g. [20, 80, 320, 1280] from dataset.level_dims
    self_condition / t_dim: passed through to each VelocityNet.
    """
    def __init__(self, level_dims, h_dim=256, n_layers=4, self_condition=False, t_dim=64):
        super().__init__()
        self.self_condition = self_condition
        self.level_dims = level_dims
        self.nets = nn.ModuleList([VelocityNet(d, h_dim, n_layers, self_condition=self_condition, t_dim=t_dim)
                                   for d in level_dims])

    def forward(self, x, t, x_self_cond=None):
        outs, offset = [], 0
        for net, d in zip(self.nets, self.level_dims):
            sc_slice = x_self_cond[:, offset:offset+d] if x_self_cond is not None else None
            outs.append(net(x[:, offset:offset+d], t, sc_slice))
            offset += d
        return torch.cat(outs, dim=1)


In [ ]:
#| export
class CrossLevelFlowModel(nn.Module):
    """Flow model with cross-level attention for joint velocity prediction.
    Each level is projected to h_dim, t is embedded sinusoidally and added to every token,
    a small transformer cross-attends across all levels (so L3 attends to L0-L2 without
    cascading), per-level residual MLPs refine, per-level heads decode velocity.
    Sequence length = n_levels (typically 4) so attention cost is negligible.
    Uses norm_first=True (pre-LN) for training stability.
    self_condition: same 50%-dropout self-conditioning as VelocityNet.
    t_dim: sinusoidal time embedding dim, projected to h_dim and added to each level token.
    """
    def __init__(self, level_dims, h_dim=512, n_layers=4, n_attn_layers=2, n_heads=8,
                 self_condition=False, t_dim=64):
        super().__init__()
        self.self_condition = self_condition
        self.level_dims = level_dims
        self.t_dim = t_dim
        in_mul = 2 if self_condition else 1
        # Per-level input projections: [x_level, (x_sc)] → h_dim  (t added separately)
        self.level_in  = nn.ModuleList([nn.Linear(d * in_mul, h_dim) for d in level_dims])
        # Time embedding: sinusoidal t_dim → h_dim (added to every level token)
        self.t_proj = nn.Sequential(nn.Linear(t_dim, h_dim), nn.SiLU(), nn.Linear(h_dim, h_dim))
        # Cross-level transformer (seq_len = n_levels ≈ 4; norm_first=True for stability)
        enc_layer = nn.TransformerEncoderLayer(h_dim, n_heads, dim_feedforward=h_dim * 4,
                                               batch_first=True, dropout=0.0, norm_first=True)
        self.cross_attn = nn.TransformerEncoder(enc_layer, num_layers=n_attn_layers)
        # Per-level residual MLPs (operate in h_dim space)
        self.level_mlp = nn.ModuleList([
            nn.ModuleList([nn.Linear(h_dim, h_dim) for _ in range(n_layers - 1)])
            for _ in level_dims])
        # Per-level output heads → velocity
        self.level_out = nn.ModuleList([nn.Linear(h_dim, d) for d in level_dims])

    def forward(self, x, t, x_self_cond=None):
        if t.dim() > 1: t = t.squeeze(-1)
        elif t.dim() == 0: t = t.unsqueeze(0).expand(x.size(0))
        t_emb = self.t_proj(sinusoidal_time_emb(t, self.t_dim))   # [B, h_dim]
        # Build per-level tokens
        tokens, offset = [], 0
        for proj, d in zip(self.level_in, self.level_dims):
            xd = x[:, offset:offset+d]
            if self.self_condition:
                sc = x_self_cond[:, offset:offset+d] if x_self_cond is not None else torch.zeros_like(xd)
                inp = torch.cat([xd, sc], dim=1)
            else:
                inp = xd
            tokens.append(F.gelu(proj(inp)) + t_emb)   # add time additively
            offset += d
        # Cross-attend across levels: [B, n_levels, h_dim]
        tokens = self.cross_attn(torch.stack(tokens, dim=1))
        # Per-level residual MLP + output head
        outs = []
        for mlp_layers, out_proj, tok in zip(self.level_mlp, self.level_out, tokens.unbind(1)):
            h = tok
            for layer in mlp_layers:
                h = F.gelu(layer(h)) + h
            outs.append(out_proj(h))
        return torch.cat(outs, dim=1)


In [ ]:
#| export
class FiLM(nn.Module):
    """Feature-wise Linear Modulation with pre-norm (AdaLN style).

    Applies adaptive LayerNorm: output = (1 + γ(cond)) * LN(x) + β(cond).
    Weights initialised to zero so the module starts as a plain LayerNorm
    (identity modulation), giving stable early training.
    """
    def __init__(self, cond_dim, feat_dim):
        super().__init__()
        self.norm  = nn.LayerNorm(feat_dim)
        self.gamma = nn.Linear(cond_dim, feat_dim)
        self.beta  = nn.Linear(cond_dim, feat_dim)
        nn.init.zeros_(self.gamma.weight); nn.init.zeros_(self.gamma.bias)
        nn.init.zeros_(self.beta.weight);  nn.init.zeros_(self.beta.bias)

    def forward(self, x, cond):
        return (1 + self.gamma(cond)) * self.norm(x) + self.beta(cond)


class ConditionalFineFlowModel(nn.Module):
    """Per-patch flow model for fine levels (L3, L4, L5).

    The original single-token-per-level design projected the entire L4 level
    (256 patches × 5 comp = 1280d) through one Linear into h_dim, losing all
    per-patch identity.  Every patch received the same velocity.

    This model treats each patch as its own token and processes them with
    *shared* MLP weights — like a per-patch MLP applied in parallel.
    Conditioning comes from spatially-aligned parent tokens at each coarse level,
    looked up via precomputed parent indices (registered as buffers).  No
    within-level attention is needed: each patch is independent given its parents.

    Args:
        cond_dims:     flattened PCA dims for coarse levels, e.g. [18, 96, 512]
        target_dims:   flattened PCA dims for fine levels,   e.g. [1280, 1280, 3072]
        target_n_comp: PCA components per fine patch,        e.g. [20, 5, 3]
        h_dim:         shared hidden dim
        n_layers:      number of pre-norm residual MLP blocks
        t_dim:         sinusoidal time embedding dim
        cond_n_comp:   PCA components per coarse patch — int (uniform) or list (per-level)
    """
    def __init__(self, cond_dims, target_dims, target_n_comp,
                 h_dim=256, n_layers=4, t_dim=64, cond_n_comp=[18, 24, 32]):
        super().__init__()
        self.cond_dims     = list(cond_dims)
        self.target_dims   = list(target_dims)
        self.target_n_comp = list(target_n_comp)
        # cond_n_comp may be int (uniform) or list (per coarse level)
        if isinstance(cond_n_comp, (int, float)):
            cond_n_comp = [int(cond_n_comp)] * len(cond_dims)
        self.cond_n_comp   = list(cond_n_comp)
        self.t_dim         = t_dim

        cond_n_patches   = [max(1,d // nc) for d, nc in zip(cond_dims, self.cond_n_comp)]
        target_n_patches = [max(1,d // nc) for d, nc in zip(target_dims, target_n_comp)]
        cond_grids       = [self._square_grid(n) for n in cond_n_patches]
        target_grids     = [self._square_grid(n) for n in target_n_patches]
        self.target_n_patches = target_n_patches
        self._cond_n_patches  = cond_n_patches

        # Precompute parent indices: pidx_{fi}_{ki} shape [n_fine_patches]
        for fi, (fH, fW) in enumerate(target_grids):
            for ki, (cH, cW) in enumerate(cond_grids):
                self.register_buffer(f'pidx_{fi}_{ki}', self._parent_idx(fH, fW, cH, cW))

        # Time embedding
        self.t_proj = nn.Sequential(
            nn.Linear(t_dim, h_dim), nn.SiLU(), nn.Linear(h_dim, h_dim))

        # Per-fine-level: project own patch [nc] → h_dim (shared across patches)
        self.patch_in = nn.ModuleList([nn.Linear(nc, h_dim) for nc in target_n_comp])

        # Per-fine-level × per-coarse-level: project parent [cnc] → h_dim (per-level cnc)
        self.parent_proj = nn.ModuleList([
            nn.ModuleList([nn.Linear(cnc, h_dim) for cnc in self.cond_n_comp])
            for _ in target_dims])

        # Learnable per-patch positional embeddings (zero-init = no bias at start)
        self.pos_emb = nn.ParameterList([
            nn.Parameter(torch.zeros(np, h_dim)) for np in target_n_patches])

        # Pre-norm residual MLP blocks, shared across all patches within a level
        self.mlp_blocks = nn.ModuleList([
            nn.ModuleList([
                nn.Sequential(
                    nn.LayerNorm(h_dim),
                    nn.Linear(h_dim, h_dim * 4),
                    nn.GELU(),
                    nn.Linear(h_dim * 4, h_dim))
                for _ in range(n_layers)])
            for _ in target_dims])

        # Output head: h_dim → velocity [nc] per patch
        self.patch_out = nn.ModuleList([nn.Linear(h_dim, nc) for nc in target_n_comp])

    @staticmethod
    def _square_grid(n):
        s = int(round(n ** 0.5))
        assert s * s == n, f"n_patches={n} is not a perfect square"
        return s, s

    @staticmethod
    def _parent_idx(fH, fW, cH, cW):
        """Index into coarse grid (cH×cW) for every patch in fine grid (fH×fW)."""
        r = torch.arange(fH).repeat_interleave(fW)
        c = torch.arange(fW).repeat(fH)
        return (r * cH // fH) * cW + (c * cW // fW)

    def forward(self, x_target, t, x_cond):
        """
        x_target: [B, sum(target_dims)]  noisy fine-level PCA embeddings
        t:        [B] or [B,1]           timestep
        x_cond:   [B, sum(cond_dims)]    coarse predicted endpoint x1 from first-stage flow
        Returns:  [B, sum(target_dims)]  predicted velocity
        """
        B = x_target.size(0)
        if t.dim() > 1: t = t.squeeze(-1)

        t_emb = self.t_proj(sinusoidal_time_emb(t, self.t_dim))  # [B, h_dim]

        # Parse coarse levels → list of [B, n_coarse_patches, cnc]
        coarse, offset = [], 0
        for d, np, cnc in zip(self.cond_dims, self._cond_n_patches, self.cond_n_comp):
            coarse.append(x_cond[:, offset:offset+d].reshape(B, np, cnc))
            offset += d

        velocities, offset = [], 0
        for fi, (d, nc, np) in enumerate(
                zip(self.target_dims, self.target_n_comp, self.target_n_patches)):
            patches = x_target[:, offset:offset+d].reshape(B, np, nc)
            offset += d

            # Own patch + position + time  →  [B, np, h_dim]
            h = F.gelu(self.patch_in[fi](patches))
            h = h + self.pos_emb[fi]       # [B, np, h_dim]  (broadcast over B)
            h = h + t_emb.unsqueeze(1)     # [B, np, h_dim]  (broadcast over np)

            # Parent context from each coarse level (single gather per level)
            for ki, (coarse_emb, proj) in enumerate(zip(coarse, self.parent_proj[fi])):
                pidx = getattr(self, f'pidx_{fi}_{ki}')       # [np]
                h = h + F.gelu(proj(coarse_emb[:, pidx, :]))  # [B, np, h_dim]

            # Shared residual MLP blocks
            for block in self.mlp_blocks[fi]:
                h = h + block(h)

            velocities.append(self.patch_out[fi](h).reshape(B, -1))

        return torch.cat(velocities, dim=1)

In [ ]:
#| export
def warp_time(t, s=0.5):
    """Parametric time warping (Scott H. Hawley, 'Flow With What You Know', ICLR 2025).
    s=1 → linear; s<1 → slower near middle; s=1.5 ≈ cosine schedule.
    Works on scalar, 1-D or 2-D tensors."""
    return 4*(1-s)*t**3 + 6*(s-1)*t**2 + (3-2*s)*t

In [ ]:
#| export
@torch.no_grad()
def rk4_step(model, y, t, dt):
    """4th-order Runge-Kutta step for the learned velocity field."""
    t_  = torch.full((y.size(0), 1), t, device=y.device, dtype=y.dtype)
    k1 = model(y,             t_)
    k2 = model(y + dt*k1/2,   t_ + dt/2)
    k3 = model(y + dt*k2/2,   t_ + dt/2)
    k4 = model(y + dt*k3,     t_ + dt)
    return y + (dt/6)*(k1 + 2*k2 + 2*k3 + k4)

@torch.no_grad()
def euler_step(model, y, t, dt):
    t_ = torch.full((y.size(0), 1), t, device=y.device, dtype=y.dtype)
    return y + model(y, t_) * dt

In [ ]:
#| export
def sample_source(shape, device='cpu', source_df=None, source_scales=None, level_dims=None):
    """Sample from source distribution with optional per-level Student-t and scaling.
    source_df: scalar df → Student-t for all dims; list → per-level (None/0 = Gaussian, float = Student-t)
    source_scales: list of per-level scale factors applied after sampling
    """
    if isinstance(source_df, (list, tuple)):
        # Per-level: each slice sampled independently
        assert level_dims is not None, "level_dims required for per-level source_df"
        batch = shape[:-1]
        y = torch.empty(*shape, device=device)
        offset = 0
        for df, d in zip(source_df, level_dims):
            sl = (*batch, d)
            if df:
                normal = torch.randn(*sl, device=device)
                gamma  = torch._standard_gamma(torch.full(sl, df/2, device=device)) / (df/2)
                y[..., offset:offset+d] = normal / gamma.sqrt()
            else:
                y[..., offset:offset+d] = torch.randn(*sl, device=device)
            offset += d
    elif source_df:
        # Scalar df → Student-t for all dims
        normal = torch.randn(*shape, device=device)
        gamma  = torch._standard_gamma(torch.full(shape, source_df/2, device=device)) / (source_df/2)
        y = normal / gamma.sqrt()
    else:
        y = torch.randn(*shape, device=device)
    if source_scales is not None and level_dims is not None:
        offset = 0
        for scale, d in zip(source_scales, level_dims):
            y[..., offset:offset+d] *= scale
            offset += d
    return y

In [ ]:
#| export
@torch.no_grad()
def generate_samples_conditional(coarse_model, fine_model, n_samples, coarse_dim,
                                 target_dims, device='cpu', n_steps=20,
                                 step_fn=rk4_step, warp_s=0.5,
                                 coarse_source_df=None, coarse_source_scales=None,
                                 coarse_level_dims=None, fine_source_scales=None):
    """Two-stage conditional sampler: coarse flow → fine conditional flow.

    Conditioning signal: x1_pred_coarse = x_t_coarse + (1-t) * v_coarse
    — the coarse model's predicted endpoint at each step.  This is more
    informative than the noisy state x_t_coarse alone (especially at small t
    where x_t is mostly noise), and is in the same space as the final coarse
    embeddings, normalising out the t-dependence of the velocity scale.

    Both models share the *same* timestep grid — enforced by construction.
    The coarse model is called once per step to get v_coarse for x1_pred,
    then step_fn (which may call it again internally for RK4) advances the
    coarse state.  The fine model uses a simple Euler step conditioned on
    x1_pred_coarse computed at the start of each step.

    Training counterpart: at each batch, freeze coarse model, compute
    x1_pred_coarse = x_t_coarse + (1-t)*coarse_model(x_t_coarse, t),
    then train fine_model(x_t_fine, t, x1_pred_coarse) with flow-matching loss.

    Args:
        coarse_model:        first-stage flow (CrossLevelFlowModel / PerLevelFlowModel)
        fine_model:          ConditionalFineFlowModel
        coarse_dim:          total dim of coarse-level output (sum of PCA dims L0-L3)
        target_dims:         list of flattened dims for fine levels (e.g. [D_L4, D_L5])
        coarse_source_*:     source distribution kwargs forwarded to the coarse stage
        fine_source_scales:  optional per-level scale factors for fine-level noise
    Returns:
        coarse_out : [n_samples, coarse_dim]
        fine_out   : [n_samples, sum(target_dims)]
    """
    fine_dim = sum(target_dims)
    y_coarse = sample_source((n_samples, coarse_dim), device=device,
                             source_df=coarse_source_df,
                             source_scales=coarse_source_scales,
                             level_dims=coarse_level_dims)
    y_fine = sample_source((n_samples, fine_dim), device=device,
                           source_scales=fine_source_scales,
                           level_dims=target_dims)
    ts = warp_time(torch.linspace(0, 1, n_steps + 1), s=warp_s)
    coarse_model.eval(); fine_model.eval()
    for i in range(n_steps):
        dt   = (ts[i+1] - ts[i]).item()
        t_s  = ts[i].item()
        t    = torch.full((n_samples, 1), t_s, device=device)

        # x1_pred_coarse: coarse model's predicted endpoint at current state/time
        v_coarse_pred  = coarse_model(y_coarse, t)
        x1_pred_coarse = y_coarse + (1 - t_s) * v_coarse_pred  # [n_samples, coarse_dim]

        # Advance coarse state (step_fn may call coarse_model again internally for RK4)
        y_coarse = step_fn(coarse_model, y_coarse, t_s, dt)

        # Fine model conditioned on x1_pred_coarse; Euler step
        v_fine = fine_model(y_fine, t, x1_pred_coarse)
        y_fine = y_fine + v_fine * dt

    return y_coarse, y_fine

In [ ]:
#| eval: false
# Smoke test: ConditionalFineFlowModel forward pass
# cond: 3 coarse levels with 20 comp/patch → [1,4,16] patches (1×1, 2×2, 4×4 grids)
# target: 2 fine levels with [4,3] comp/patch → [16,64] patches (4×4, 8×8 grids)
import torch
cond_dims    = [20, 80, 320]          # 1,4,16 coarse patches × 20 comp
target_dims  = [64, 192]              # 16,64 fine patches × 4,3 comp
target_n_comp = [4, 3]
m = ConditionalFineFlowModel(cond_dims, target_dims, target_n_comp, h_dim=64, n_layers=3)
print(f'Params: {sum(p.numel() for p in m.parameters()):,}')
B = 4
v = m(torch.randn(B, sum(target_dims)), torch.rand(B), torch.randn(B, sum(cond_dims)))
assert v.shape == (B, sum(target_dims)), f"Expected {(B, sum(target_dims))}, got {v.shape}"
print(f'Output shape: {v.shape}  OK')
# Verify parent indices (L1 has 4 patches in 2×2 grid; L2 fine patch 0 at (0,0) → parent (0,0)=0)
assert m.pidx_1_1[0].item() == 0
print('Parent index check OK')

In [ ]:
#| export
def ann_repair(source, target, n_projections=1, chunk_size=None):
    """Approximate nearest-neighbor re-pairing of source and target batches.

    Sorts both source and target by their projection onto random unit vectors
    and pairs by rank — equivalent to exact 1-D OT along that direction.
    Fully on-device (GPU-friendly), O(B log B) per projection.

    chunk_size: if set, processes the batch in chunks of this size and repairs
    independently within each chunk. Smaller chunks are faster but less optimal;
    default (None) processes the whole batch at once.

    With n_projections > 1, tries multiple random directions and keeps the
    pairing with the lowest total squared transport cost.

    Args:
        source:        (B, D) tensor on any device
        target:        (B, D) tensor on same device
        n_projections: number of random projections to try per chunk
        chunk_size:    chunk size for within-batch processing (None = full batch)

    Returns:
        (source_repaired, target_repaired): re-ordered so source[i] ↔ target[i]
        approximately minimises total squared transport cost.
    """
    B, D = source.shape
    C = B if (chunk_size is None or chunk_size >= B) else chunk_size

    s_out = torch.empty_like(source)
    t_out = torch.empty_like(target)
    for start in range(0, B, C):
        end  = min(start + C, B)
        s, t = source[start:end], target[start:end]
        best_s, best_t, best_cost = s, t, float('inf')
        for _ in range(n_projections):
            proj   = torch.randn(D, device=s.device, dtype=s.dtype)
            proj   = proj / proj.norm()
            s_rep  = s[(s @ proj).argsort()]
            t_rep  = t[(t @ proj).argsort()]
            cost   = (s_rep - t_rep).pow(2).sum().item()
            if cost < best_cost:
                best_cost = cost
                best_s, best_t = s_rep, t_rep
        s_out[start:end] = best_s
        t_out[start:end] = best_t
    return s_out, t_out


In [ ]:
#| export
@torch.no_grad()
def generate_samples(model, n_samples, dim, device='cpu',
                     n_steps=20, step_fn=rk4_step, warp_s=0.5, source_df=None,
                     source_scales=None, level_dims=None):
    """Sample from the flow model: integrate noise → embedding space."""
    y = sample_source((n_samples, dim), device=device, source_df=source_df,
                      source_scales=source_scales, level_dims=level_dims)
    ts = torch.linspace(0, 1, n_steps + 1)
    ts = warp_time(ts, s=warp_s)
    model.eval()
    for i in range(n_steps):
        dt = (ts[i+1] - ts[i]).item()
        y  = step_fn(model, y, ts[i].item(), dt)
    return y

In [ ]:
#| export
def mmd_rbf(x, y, n_sub=2000):
    """Unbiased MMD² with RBF kernel, median bandwidth heuristic.
    x, y: (N, D) tensors. Subsamples to n_sub for speed."""
    if x.size(0) > n_sub: x = x[torch.randperm(x.size(0))[:n_sub]]
    if y.size(0) > n_sub: y = y[torch.randperm(y.size(0))[:n_sub]]
    xy = torch.cat([x, y], dim=0)
    sigma2 = torch.cdist(xy, xy).median().pow(2).clamp(min=1e-6)
    def rbf(a, b): return torch.exp(-torch.cdist(a, b).pow(2) / (2 * sigma2))
    return (rbf(x, x).mean() + rbf(y, y).mean() - 2 * rbf(x, y).mean()).item()


In [ ]:

#| export
def wasserstein_score(x, y, n_projections=200, n_sub=2000):
    """Sliced Wasserstein distance: average 1-D Wasserstein over random projections.
    Falls back gracefully if geomloss is unavailable.
    Returns nan on numerical failure (overflow, diverged samples, etc.).
    x, y: (N, D) numpy arrays."""
    try:
        import geomloss
        loss = geomloss.SamplesLoss("sinkhorn", p=2, blur=0.05)
        xt = torch.tensor(x[:n_sub]).float()
        yt = torch.tensor(y[:n_sub]).float()
        return loss(xt, yt).item()
    except ImportError:
        pass
    except Exception:
        return float('nan')
    try:
        from scipy.stats import wasserstein_distance
        rng = np.random.default_rng(0)
        D = x.shape[1]
        projs = rng.standard_normal((D, n_projections))
        projs /= np.linalg.norm(projs, axis=0, keepdims=True)
        px, py = x[:n_sub] @ projs, y[:n_sub] @ projs
        return float(np.mean([wasserstein_distance(px[:, i], py[:, i]) for i in range(n_projections)]))
    except Exception:
        return float('nan')


In [ ]:

#| export
@torch.no_grad()
def eval_flow(model, real_embeddings, n_samples=10000, n_steps=20, warp_s=0.5, device='cpu',
              source_df=None, source_scales=None, level_dims=None, gen=None, level_names=None):
    """Compare distributional statistics of real vs generated embeddings, per level.
    Returns flat dict with keys like 'L0/mmd', 'L0/wasserstein', 'L0/real_std', etc.
    Also returns global 'mmd' and 'wasserstein' for backward compatibility.
    real_embeddings: (N, D) tensor.
    gen: optional pre-computed generated samples (N, D) tensor — skips generate_samples().
    level_names: optional list of strings e.g. ['L4', 'L5'] to override default 'L0', 'L1' keys.
    """
    from scipy.stats import skew, kurtosis
    idx = torch.randperm(real_embeddings.size(0))[:n_samples]
    real = real_embeddings[idx].float()
    if gen is None:
        dim = real_embeddings.shape[1]
        gen = generate_samples(model, n_samples, dim, device=device,
                               n_steps=n_steps, warp_s=warp_s, source_df=source_df,
                               source_scales=source_scales, level_dims=level_dims).cpu()
    else:
        gen = gen[:n_samples].float().cpu()
    r, g = real.numpy(), gen.numpy()

    metrics = {}
    # Global stats
    metrics['real_mean']  = float(r.mean())
    metrics['real_std']   = float(r.std())
    metrics['real_skew']  = float(skew(r.ravel()))
    metrics['real_kurt']  = float(kurtosis(r.ravel()))
    metrics['gen_mean']   = float(g.mean())
    metrics['gen_std']    = float(g.std())
    metrics['gen_skew']   = float(skew(g.ravel()))
    metrics['gen_kurt']   = float(kurtosis(g.ravel()))
    metrics['mmd']        = mmd_rbf(real, gen)
    metrics['wasserstein'] = wasserstein_score(r, g)

    # Per-level stats
    if level_dims is not None:
        offset = 0
        for i, d in enumerate(level_dims):
            rl = r[:, offset:offset+d]
            gl = g[:, offset:offset+d]
            rt, gt = torch.tensor(rl), torch.tensor(gl)
            lname = level_names[i] if level_names else f'L{i}'
            metrics[f'{lname}/real_std']    = float(rl.std())
            metrics[f'{lname}/gen_std']     = float(gl.std())
            metrics[f'{lname}/real_kurt']   = float(kurtosis(rl.ravel()))
            metrics[f'{lname}/gen_kurt']    = float(kurtosis(gl.ravel()))
            metrics[f'{lname}/mmd']         = mmd_rbf(rt, gt)
            metrics[f'{lname}/wasserstein'] = wasserstein_score(rl, gl)
            offset += d

    w = max(len(k) for k in metrics)
    for k, v in metrics.items():
        print(f'  {k:{w}s} = {v:.4f}')
    return metrics

In [ ]:
#| export
def eval_jacobian_norm_vs_t(model, x_sample, n_t=20, n_epsilon=4, cond=None, device='cpu'):
    """Estimate Frobenius norm of the Jacobian dv/dx as a function of t.

    Uses Hutchinson estimator: E_ε[||J^T ε||²] = ||J||²_F  with Rademacher ε.
    A peak in the curve at some t* reveals where the flow is making its hardest
    topological decisions (routing mass to disjoint clusters).

    Args:
        model:     velocity field; called as model(x, t) or model(x, t, cond)
        x_sample:  (B, D) batch of real data points
        n_t:       number of time steps to sweep
        n_epsilon: number of Rademacher samples per t (more = lower variance)
        cond:      optional conditioning tensor (B, cond_dim) for conditional models
        device:    compute device
    Returns:
        t_vals (list[float]), norm_vals (list[float])
    """
    model.eval()
    x_sample = x_sample.to(device).float()
    t_vals, norm_vals = [], []

    for t_val in torch.linspace(0.01, 0.99, n_t).tolist():
        t = torch.full((len(x_sample), 1), t_val, device=device)
        x = x_sample.detach().requires_grad_(True)

        est_list = []
        for _ in range(n_epsilon):
            epsilon = (torch.randint_like(x, low=0, high=2).float() * 2 - 1)  # Rademacher ±1
            if cond is not None:
                v = model(x, t, cond.to(device))
            else:
                v = model(x, t)
            vjp = torch.autograd.grad(v, x, grad_outputs=epsilon,
                                       retain_graph=True, create_graph=False)[0]
            # Frobenius norm estimator: E[||J^T ε||²] = ||J||²_F
            est_list.append(vjp.pow(2).sum(dim=-1))   # (B,)

        norm_sq = torch.stack(est_list).mean(0).mean().item()   # scalar
        t_vals.append(t_val)
        norm_vals.append(norm_sq ** 0.5)              # sqrt for Frobenius norm

    return t_vals, norm_vals

In [ ]:

#| export
@torch.no_grad()
def plot_level_histograms(model, real_embeddings, level_dims, n_samples=10000,
                          n_steps=20, warp_s=0.5, device='cpu', n_bins=100, source_df=None,
                          source_scales=None, epoch=None, gen=None, level_names=None):
    """Return dict of per-level histogram figures {'L0': fig, 'L1': fig, ...}.
    level_dims: list of ints, flattened PCA dims per level e.g. [20, 80, 320, 1280]
    real_embeddings: (N, sum(level_dims)) tensor
    gen: optional pre-computed generated samples — skips generate_samples().
    level_names: optional list of strings e.g. ['L4', 'L5'] to override default 'L0', 'L1' labels.
    """
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    idx = torch.randperm(real_embeddings.size(0))[:n_samples]
    real = real_embeddings[idx].float().numpy()
    if gen is None:
        dim = real_embeddings.shape[1]
        gen = generate_samples(model, n_samples, dim, device=device,
                               n_steps=n_steps, warp_s=warp_s, source_df=source_df,
                               source_scales=source_scales, level_dims=level_dims).cpu().numpy()
    else:
        gen = gen[:n_samples].float().cpu().numpy()
    figs = {}
    offset = 0
    for i, d in enumerate(level_dims):
        r = real[:, offset:offset+d].ravel()
        g = gen[:,  offset:offset+d].ravel()
        lim = np.percentile(np.abs(np.concatenate([r, g])), 99)
        bins = np.linspace(-lim, lim, n_bins + 1)
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.hist(r, bins=bins, alpha=0.5, color='steelblue', label='real', density=True)
        ax.hist(g, bins=bins, alpha=0.5, color='darkorange', label='gen',  density=True)
        lname = level_names[i] if level_names else f'L{i}'
        title = f'{lname} ({d}d)'
        if epoch is not None: title += f' — Epoch {epoch}'
        ax.set_title(title)
        ax.set_xlabel('value')
        ax.legend(fontsize=8)
        plt.tight_layout()
        figs[lname] = fig
        offset += d
    return figs

In [ ]:
#| export
@torch.no_grad()
def plot_level_scatter(model, real_embeddings, level_dims, n_samples=5000,
                       n_steps=20, warp_s=0.5, device='cpu',
                       source_df=None, source_scales=None, epoch=None,
                       gen=None, level_names=None, level_n_components=None):
    """Return dict of per-level 3D PCA scatter plots {'L0/real': fig, 'L0/gen': fig, ...}.
    gen: optional pre-computed generated samples — skips generate_samples().
    level_names: optional list of strings e.g. ['L4', 'L5'] to override default 'L0', 'L1' labels.
    level_n_components: optional list of ints (one per level). If provided, each level's data is
        reshaped from (B, n_patches × n_comp) → (B × n_patches, n_comp) before PCA so that each
        patch is a point — matching the encoder training viz style (make_emb_viz / _gather_level).
        After reshape, subsampled back to n_samples points to keep plots manageable.
    """
    from midi_rae.viz import pca_project, plot_embeddings_3d
    idx = torch.randperm(real_embeddings.size(0))[:n_samples]
    real = real_embeddings[idx].float()
    if gen is None:
        dim = real_embeddings.shape[1]
        gen = generate_samples(model, n_samples, dim, device=device,
                               n_steps=n_steps, warp_s=warp_s, source_df=source_df,
                               source_scales=source_scales, level_dims=level_dims).cpu()
    else:
        gen = gen[:n_samples].float().cpu()
    figs = {}
    offset = 0
    for i, d in enumerate(level_dims):
        r = real[:, offset:offset+d]
        g = gen[:,  offset:offset+d]
        lname = level_names[i] if level_names else f'L{i}'
        title_sfx = f' — Epoch {epoch}' if epoch is not None else ''
        if level_n_components is not None:
            n_comp = level_n_components[i]
            n_patches = d // n_comp
            r = r.reshape(-1, n_comp)   # (B × n_patches, n_comp) — each patch is a point
            g = g.reshape(-1, n_comp)
            # subsample after reshape so scatter stays at ~n_samples points
            if r.shape[0] > n_samples:
                sub = torch.randperm(r.shape[0])[:n_samples]
                r = r[sub]
                g = g[sub]
        r3 = pca_project(r)
        g3 = pca_project(g)
        if r3 is not None: figs[f'{lname}/real'] = plot_embeddings_3d(r3, color_by='random', title=f'{lname} ({n_comp if level_n_components else d}d/patch) real{title_sfx}')
        if g3 is not None: figs[f'{lname}/gen']  = plot_embeddings_3d(g3, color_by='random', title=f'{lname} ({n_comp if level_n_components else d}d/patch) gen{title_sfx}')
        offset += d
    return figs

In [ ]:
#| export
def _wandb_log_viz(log_dict, eval_model, embeddings, level_dims, epoch,
                   real_scatter_logged, gen=None, level_names=None,
                   device='cpu', warp_s=0.5, source_df=None, source_scales=None,
                   level_n_components=None):
    """Log per-level histograms and 3-D scatter plots to W&B via log_dict.
    gen: optional pre-computed generated samples — avoids redundant forward passes.
    level_n_components: passed to plot_level_scatter to show per-patch points (encoder viz style).
    Modifies log_dict in-place. Returns updated real_scatter_logged flag."""
    import wandb, matplotlib.pyplot as plt
    figs = plot_level_histograms(eval_model, embeddings, level_dims, epoch=epoch,
                                 gen=gen, level_names=level_names, device=device,
                                 warp_s=warp_s, source_df=source_df, source_scales=source_scales)
    for lname, fig in figs.items():
        log_dict[f'media/hist_{lname}'] = wandb.Image(fig, caption=f'Epoch {epoch}')
        plt.close(fig)
    del figs
    scatters = plot_level_scatter(eval_model, embeddings, level_dims, epoch=epoch,
                                  gen=gen, level_names=level_names, device=device,
                                  warp_s=warp_s, source_df=source_df, source_scales=source_scales,
                                  level_n_components=level_n_components)
    for lname, fig in scatters.items():
        if lname.endswith('/real') and real_scatter_logged:
            fig.data = []
            continue
        log_dict[f'media/scatter_{lname.replace("/", "_")}'] = wandb.Html(fig.to_html())
    del scatters
    return True  # real_scatter_logged

In [ ]:
#| export
def _wandb_log_jacobian(log_dict, eval_model, sample_data, n_t=20, n_epsilon=4,
                        cond=None, device='cpu'):
    """Evaluate Jacobian Frobenius norm vs t and log a line chart to W&B via log_dict.
    Modifies log_dict in-place.
    cond: optional conditioning tensor (B, cond_dim) for conditional models."""
    import wandb
    t_vals, jac_norms = eval_jacobian_norm_vs_t(
        eval_model, sample_data, n_t=n_t, n_epsilon=n_epsilon, cond=cond, device=device)
    jac_table = wandb.Table(columns=['t', 'jacobian_norm'],
                            data=[[t, n] for t, n in zip(t_vals, jac_norms)])
    log_dict['eval/jacobian_norm_vs_t'] = wandb.plot.line(
        jac_table, 't', 'jacobian_norm', title='Jacobian Frobenius Norm vs t')

In [ ]:
#| export
@torch.no_grad()
def decode_flow_to_piano_rolls(coarse_pca, fine_emb, pca_models,
                                coarse_level_dims, fine_level_dims, fine_levels_idx,
                                cfg, decoder, device, n_samples=16):
    """Decode flow-generated coarse PCA + fine embeddings directly to piano rolls (no HMEP).
    coarse_pca:  (B, sum_coarse_dims) PCA-compressed coarse embeddings
    fine_emb:    (B, sum_fine_dims) fine embeddings in PCA space (n_components per patch)
    pca_models:  dict {level_idx: sklearn PCA} — must include fine levels if fine PCA was used
    fine_levels_idx: list of level indices e.g. [4, 5]
    """
    from midi_rae.generate import build_patch_states, batch_patch_states, build_enc_out, make_grid_pos, binarize
    from midi_rae.core import PatchState

    B = min(n_samples, coarse_pca.shape[0])
    coarse_pca = coarse_pca[:B].float()
    fine_emb   = fine_emb[:B].float()

    # Inverse PCA → coarse PatchState list
    states = [build_patch_states(coarse_pca[b], pca_models, coarse_level_dims, device)
              for b in range(B)]
    all_levels = batch_patch_states(states)

    # Fine levels: inverse-PCA from compressed space → full embedding space, then build PatchStates
    offset = 0
    for j, li in enumerate(fine_levels_idx):
        d         = fine_level_dims[j]          # n_components * n_patches (PCA-compressed)
        n_patches = d // (d // (4 ** li))       # infer n_patches
        n_patches = 4 ** li
        n_comp    = d // n_patches               # PCA components per patch
        flat_pca  = fine_emb[:, offset:offset+d].reshape(B * n_patches, n_comp).cpu().numpy()
        if li in pca_models:
            flat_full = pca_models[li].inverse_transform(flat_pca)   # (B*n_patches, D_full)
        else:
            flat_full = flat_pca                                       # no PCA: use as-is
        emb = torch.tensor(flat_full, dtype=torch.float32).reshape(B, n_patches, -1).to(device)
        pos = make_grid_pos(n_patches, device)
        all_levels.append(PatchState(emb=emb, pos=pos,
                                     non_empty=torch.ones(B, n_patches, device=device),
                                     mae_mask=torch.ones(n_patches, device=device)))
        offset += d

    enc_out = build_enc_out(all_levels)
    recons  = decoder(enc_out)
    return binarize(recons)


In [ ]:
# #| export
# def train_flow(model, dataset, cfg, device='cpu'):
#     """Train flow matching model on embedding dataset.
#     All hyperparameters are read from cfg.flow.  Manages W&B init/finish internally.
#     """
#     from midi_rae.utils import save_checkpoint, load_checkpoint, EMAModel
#     fc = cfg.flow

#     # --- Hyperparameters from cfg ---
#     n_epochs          = fc.n_epochs
#     lr                = fc.lr
#     batch_size        = fc.batch_size
#     warp_s            = fc.warp_s
#     save_every        = fc.save_every
#     eval_every        = fc.eval_every
#     viz_every         = fc.get('viz_every', 25)
#     steps_per_epoch   = fc.get('steps_per_epoch', None)
#     lr_restart_epochs = fc.get('lr_restart_epochs', 500)
#     lr_warmup_frac    = fc.get('lr_warmup_frac', 0.15)
#     grad_clip         = fc.get('grad_clip', 1.0)
#     ema_eta           = fc.get('ema_eta', 0.97)
#     ema_start_epoch   = fc.get('ema_start_epoch', 100)
#     repair_every      = fc.get('repair_every', 1)
#     n_repair_projections = fc.get('n_repair_projections', 1)
#     repair_chunk_size = fc.get('repair_chunk_size', None)

#     source_scales = list(fc.source_scales) if fc.get('source_scales') else None
#     raw_df        = fc.get('source_df', None)
#     source_df     = list(raw_df) if hasattr(raw_df, '__iter__') else raw_df
#     level_dims    = getattr(dataset, 'level_dims', None)
#     if source_df and level_dims: source_df = source_df[:len(level_dims)]

#     checkpoint = os.path.expandvars(os.path.expanduser(cfg.get('checkpoint', '') or '')) or None

#     no_wandb = cfg.get('no_wandb', False)
#     use_wandb = not no_wandb   
#     if use_wandb:
#         import wandb
#         wandb.init(project=cfg.wandb.flow_project, config=dict(fc))
#         wandb.define_metric("epoch")
#         wandb.define_metric("*", step_metric="epoch")
#         if hasattr(cfg, 'tag'): wandb.run.name = f"{cfg.tag}_{wandb.run.name}"

#     # Per-patch scatter viz: n_patches at coarse level i = 4^i
#     level_n_components = None
#     if level_dims:
#         level_n_components = [d // (4**i) for i, d in enumerate(level_dims)]

#     self_cond = getattr(model, 'self_condition', False)
#     model = model.to(device)
#     ema_model = EMAModel(model, eta=ema_eta, update_every=1, dtype=torch.float32)
#     dl = DataLoader(dataset, batch_size=batch_size, shuffle=True,
#                     num_workers=2, pin_memory=(device != 'cpu'), drop_last=True)
#     dl_iter = None
#     _steps = steps_per_epoch or len(dl)
#     optimizer = optim.Adam(model.parameters(), lr=lr)
#     loss_fn   = nn.MSELoss()
#     global_step = 0
#     epoch_start = 0
#     real_scatter_logged = False
#     dim = dataset.embeddings.shape[1]

#     if checkpoint:
#         model, ckpt = load_checkpoint(model, checkpoint, return_all=True)
#         optimizer.load_state_dict(ckpt['optimizer_state_dict'])
#         epoch_start = ckpt['epoch']
#         global_step = epoch_start * _steps
#         print(f"Resumed from {checkpoint} (epoch {epoch_start})")
#         ema_model.ema.load_state_dict(model.state_dict())

#     scheduler = make_warmup_cosine_restart_scheduler(
#         optimizer, T_0=lr_restart_epochs, T_mult=2, warmup_frac=lr_warmup_frac, eta_min=1e-6)
#     for _ in range(epoch_start): scheduler.step()

#     for epoch in range(epoch_start, n_epochs):
#         model.train()
#         epoch_loss = 0.
#         if steps_per_epoch:
#             if dl_iter is None:
#                 import itertools; dl_iter = itertools.cycle(dl)
#             batches = (next(dl_iter) for _ in range(_steps))
#         else:
#             batches = dl
#         pbar = tqdm(batches, total=_steps, desc=f'Epoch {epoch+1}/{n_epochs}', leave=False)
#         for target in pbar:
#             target = target.to(device)
#             B, D   = target.shape
#             source = sample_source((B, D), device=device, source_df=source_df,
#                                    source_scales=source_scales, level_dims=level_dims)
#             if repair_every and global_step % repair_every == 0:
#                 source, target = ann_repair(source, target,
#                                             n_projections=n_repair_projections,
#                                             chunk_size=repair_chunk_size)
#             t = torch.rand(B, 1, device=device)
#             if warp_s != 1.0: t = warp_time(t, s=warp_s)
#             x_t = (1 - t) * source + t * target
#             v   = target - source
#             x_self_cond = None
#             if self_cond and torch.rand(1).item() < 0.5:
#                 with torch.no_grad():
#                     v_first = model(x_t, t, None)
#                     x_self_cond = (x_t + (1 - t) * v_first).detach()
#             optimizer.zero_grad()
#             v_pred = model(x_t, t, x_self_cond)
#             loss   = loss_fn(v_pred, v)
#             loss.backward()
#             if grad_clip > 0:
#                 torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
#             optimizer.step()
#             ema_model.update(model)
#             epoch_loss += loss.item()
#             pbar.set_postfix(loss=f'{loss.item():.4f}')
#             global_step += 1

#         scheduler.step()
#         avg_loss = epoch_loss / len(dl)
#         cur_lr = scheduler.get_last_lr()[0]
#         print(f'Epoch {epoch+1}/{n_epochs}  loss={avg_loss:.4f}  lr={cur_lr:.2e}')
#         if use_wandb: wandb.log({'train/epoch_loss': avg_loss, 'train/lr': cur_lr, 'epoch': epoch+1}, step=global_step)

#         if eval_every and (epoch + 1) % eval_every == 0:
#             print(f'  --- eval epoch {epoch+1} ---')
#             eval_model = ema_model.ema if (epoch + 1) >= ema_start_epoch else model
#             gen = generate_samples(eval_model, 10000, dim, device=device, n_steps=20,
#                                    warp_s=warp_s, source_df=source_df,
#                                    source_scales=source_scales, level_dims=level_dims).cpu()
#             metrics = eval_flow(eval_model, dataset.embeddings, device=device, warp_s=warp_s,
#                                 source_df=source_df, source_scales=source_scales,
#                                 level_dims=level_dims, gen=gen)
#             if use_wandb:
#                 log_dict = {f'eval/{k}': v for k, v in metrics.items()}
#                 if level_dims and viz_every and (epoch + 1) % viz_every == 0:
#                     real_scatter_logged = _wandb_log_viz(
#                         log_dict, eval_model, dataset.embeddings, level_dims, epoch+1,
#                         real_scatter_logged, gen=gen, device=device, warp_s=warp_s,
#                         source_df=source_df, source_scales=source_scales,
#                         level_n_components=level_n_components)
#                     _wandb_log_jacobian(log_dict, eval_model,
#                                         dataset.embeddings[:256].float().to(device))
#                     gc.collect()
#                 log_dict['epoch'] = epoch+1
#                 wandb.log(log_dict, step=global_step)
#             del gen
#             model.train()

#         save_checkpoint([model, ema_model.ema], epoch+1, avg_loss, cfg or {}, optimizer=optimizer,
#                         save_every=save_every, tag=f'flow_{model.__class__.__name__}')

#     print(f"FINISHED. Best metric: final loss={avg_loss:.4f}")
#     if use_wandb: wandb.finish()
#     return model

In [ ]:
#| export
def make_warmup_cosine_restart_scheduler(optimizer, T_0, T_mult=2, warmup_frac=0.15, eta_min=1e-6):
    """LambdaLR implementing true warm restarts: linear ramp-up → cosine decay per cycle.
    Periods double each restart (T_mult=2): T_0, T_0*2, T_0*4, ...
    warmup_frac: fraction of each cycle spent warming up to peak lr.
    eta_min: minimum lr (as absolute value, not a multiplier)."""
    base_lr = optimizer.param_groups[0]['lr']

    def get_cycle(epoch):
        """Return (cycle index, position within cycle, cycle length)."""
        T_i, T_prev = T_0, 0
        while T_prev + T_i <= epoch:
            T_prev += T_i
            T_i = int(T_i * T_mult)
        return T_prev, T_i   # cycle_start, cycle_length

    def lr_lambda(epoch):
        cycle_start, T_i = get_cycle(epoch)
        T_cur = epoch - cycle_start
        warmup_end = max(1, int(T_i * warmup_frac))
        if T_cur < warmup_end:
            return T_cur / warmup_end                              # linear warmup → 1.0 (base_lr)
        progress = (T_cur - warmup_end) / max(1, T_i - warmup_end)
        cos_val = 0.5 * (1 + math.cos(math.pi * progress))        # 1 → 0
        return eta_min / base_lr + (1 - eta_min / base_lr) * cos_val

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


In [ ]:
#| export
def train_flow_conditional(coarse_model, fine_model, dataset, cfg, device='cpu',
                           decoder=None, pca_models=None):
    """Train ConditionalFineFlowModel with a frozen coarse model as conditioning.
    All hyperparameters read from cfg.flow2.  Manages W&B init/finish internally.

    decoder, pca_models: if both provided, decode piano rolls during viz.
    """
    from midi_rae.utils import save_checkpoint, load_checkpoint, EMAModel
    from midi_rae.data import ConditionalFlowChunkSampler, ConditionalFlowDataset
    from hydra.core.hydra_config import HydraConfig
    from torchvision.utils import make_grid

    cjprint(f"config file: {HydraConfig.get().job.config_name}\nconfig: {cfg}\ndevice = {device}",color="green")

    fc = cfg.flow2

    # --- Hyperparameters from cfg ---
    n_epochs          = fc.n_epochs
    lr                = fc.lr
    batch_size        = fc.batch_size
    warp_s            = fc.warp_s
    save_every        = fc.get('save_every', 10)
    viz_every         = fc.get('viz_every', 10)
    eval_every        = min(viz_every, fc.get('eval_every', 10))
    steps_per_epoch   = fc.get('steps_per_epoch', None)
    lr_restart_epochs = fc.get('lr_restart_epochs', 500)
    lr_warmup_frac    = fc.get('lr_warmup_frac', 0.15)
    grad_clip         = fc.get('grad_clip', 1.0)
    ema_eta           = fc.get('ema_eta', 0.97)
    ema_start_epoch   = fc.get('ema_start_epoch', 100)
    repair_every      = fc.get('repair_every', 1)
    n_repair_projections = fc.get('n_repair_projections', 1)
    repair_chunk_size = fc.get('repair_chunk_size', None)
    fine_source_scales = list(fc.get('fine_source_scales', [])) or None

    fine_levels = list(fc.get('fine_levels', [4, 5]))
    fine_level_names = [f'L{l}' for l in fine_levels]
    fine_levels_idx  = fine_levels

    raw_fn = fc.get('fine_n_components', None)
    fine_n_components = (list(raw_fn) if hasattr(raw_fn, '__iter__') else int(raw_fn)) if raw_fn is not None else None
    fn_list = (fine_n_components if isinstance(fine_n_components, list)
               else [fine_n_components] * len(fine_levels)) if fine_n_components else None

    checkpoint = os.path.expandvars(os.path.expanduser(cfg.get('checkpoint', '') or '')) or None

    use_wandb = not cfg.get('no_wandb', False) and hasattr(cfg.wandb, 'flow_project')
    if use_wandb:
        wandb.init(project=cfg.wandb.flow_project, config=dict(fc))
        wandb.define_metric("epoch")
        wandb.define_metric("*", step_metric="epoch")
        if hasattr(cfg, 'tag'): wandb.run.name = f"{cfg.tag}_{wandb.run.name}"

    coarse_model = coarse_model.to(device)
    coarse_model.eval()
    for p in coarse_model.parameters(): p.requires_grad_(False)

    fine_model = fine_model.to(device)
    ema_model  = EMAModel(fine_model, eta=ema_eta, update_every=1, dtype=torch.float32)

    use_fine_pca = getattr(dataset, '_use_fine_pca', False)
    if isinstance(dataset, ConditionalFlowDataset) and not use_fine_pca:
        sampler = ConditionalFlowChunkSampler(dataset)
        dl = DataLoader(dataset, batch_size=batch_size, sampler=sampler,
                        num_workers=0, pin_memory=(device != 'cpu'), drop_last=True)
    else:
        dl = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                        num_workers=2, pin_memory=(device != 'cpu'), drop_last=True)
    dl_iter = None
    _steps  = steps_per_epoch or len(dl)

    optimizer = optim.Adam(fine_model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    scheduler = make_warmup_cosine_restart_scheduler(
        optimizer, T_0=lr_restart_epochs, T_mult=2, warmup_frac=lr_warmup_frac, eta_min=1e-6)

    global_step = 0
    epoch_start = 0
    real_scatter_logged = False

    if checkpoint:
        fine_model, ckpt = load_checkpoint(fine_model, checkpoint, return_all=True)
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        epoch_start = ckpt['epoch']
        global_step = epoch_start * _steps
        for _ in range(epoch_start): scheduler.step()
        ema_model.ema.load_state_dict(fine_model.state_dict())
        print(f"Resumed from {checkpoint} (epoch {epoch_start})")

    fine_level_dims   = dataset.fine_level_dims
    coarse_dim        = dataset.coarse.shape[1]
    coarse_level_dims = dataset.coarse_level_dims
    coarse_level_names = [f'L{i}' for i in range(len(coarse_level_dims))]

    def _get_eval_fine(n=2000):
        if getattr(dataset, '_use_fine_pca', False):
            return dataset.fine[:n].float()
        dataset._load_fine_chunk(0)
        return dataset._fine_chunk_data[:n].float()

    for epoch in range(epoch_start, n_epochs):
        fine_model.train()
        epoch_loss = 0.
        if steps_per_epoch:
            if dl_iter is None:
                import itertools; dl_iter = itertools.cycle(dl)
            batches = (next(dl_iter) for _ in range(_steps))
        else:
            batches = dl
        pbar = tqdm(batches, total=_steps, desc=f'Epoch {epoch+1}/{n_epochs}', leave=False)
        for real_coarse, real_fine in pbar:
            real_coarse = real_coarse.to(device)
            real_fine   = real_fine.to(device)
            B = real_coarse.size(0)

            t = warp_time(torch.rand(B, 1, device=device), s=warp_s)
            noise_coarse = torch.randn_like(real_coarse)
            noise_fine   = sample_source((B, real_fine.size(1)), device=device,
                                         source_scales=fine_source_scales,
                                         level_dims=fine_level_dims)

            if repair_every and global_step % repair_every == 0:
                src_cat = torch.cat([noise_coarse, noise_fine], dim=1)
                tgt_cat = torch.cat([real_coarse,  real_fine],  dim=1)
                src_cat, tgt_cat = ann_repair(src_cat, tgt_cat,
                                              n_projections=n_repair_projections,
                                              chunk_size=repair_chunk_size)
                noise_coarse = src_cat[:, :noise_coarse.size(1)]
                noise_fine   = src_cat[:, noise_coarse.size(1):]
                real_coarse  = tgt_cat[:, :real_coarse.size(1)]
                real_fine    = tgt_cat[:, real_coarse.size(1):]

            x_t_coarse = (1 - t) * noise_coarse + t * real_coarse
            x_t_fine   = (1 - t) * noise_fine   + t * real_fine
            v_fine_target = real_fine - noise_fine

            with torch.no_grad():
                v_coarse       = coarse_model(x_t_coarse, t)
                x1_pred_coarse = x_t_coarse + (1 - t) * v_coarse

            optimizer.zero_grad()
            v_pred = fine_model(x_t_fine, t, x1_pred_coarse)
            loss   = loss_fn(v_pred, v_fine_target)
            loss.backward()
            if grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(fine_model.parameters(), max_norm=grad_clip)
            optimizer.step()
            ema_model.update(fine_model)

            epoch_loss += loss.item()
            pbar.set_postfix(loss=f'{loss.item():.4f}')
            global_step += 1

        scheduler.step()
        avg_loss = epoch_loss / _steps
        cur_lr   = scheduler.get_last_lr()[0]
        print(f'Epoch {epoch+1}/{n_epochs}  loss={avg_loss:.4f}  lr={cur_lr:.2e}')

        log_dict = {'train/loss': avg_loss, 'train/lr': cur_lr, 'epoch': epoch+1}

        if eval_every and (epoch + 1) % eval_every == 0:
            eval_model = ema_model.ema if (epoch + 1) >= ema_start_epoch else fine_model
            gen_coarse, gen_fine = generate_samples_conditional(
                coarse_model, eval_model,
                n_samples=2000, coarse_dim=coarse_dim,
                target_dims=fine_level_dims, device=device,
                n_steps=20, warp_s=warp_s,
                coarse_level_dims=coarse_level_dims,
                fine_source_scales=fine_source_scales,
            )
            real_fine_eval = _get_eval_fine(2000)
            gen_fine_cpu   = gen_fine.cpu()
            gen_coarse_cpu = gen_coarse.cpu()
            real_coarse_eval = dataset.coarse[:2000].float()

            # Fine-level metrics
            metrics = eval_flow(eval_model, real_fine_eval,
                                n_samples=2000, level_dims=fine_level_dims,
                                gen=gen_fine_cpu, level_names=fine_level_names)
            log_dict.update({f'eval/{k}': v for k, v in metrics.items()})

            # Coarse-level metrics (frozen model, no grad)
            coarse_metrics = eval_flow(coarse_model, real_coarse_eval,
                                       n_samples=2000, level_dims=coarse_level_dims,
                                       gen=gen_coarse_cpu, level_names=coarse_level_names)
            log_dict.update({f'eval/{k}': v for k, v in coarse_metrics.items()})

            if (wandb.run is not None) and viz_every and (epoch + 1) % viz_every == 0:
                real_scatter_logged = _wandb_log_viz(
                    log_dict, eval_model, real_fine_eval, fine_level_dims, epoch+1,
                    real_scatter_logged, gen=gen_fine_cpu, level_names=fine_level_names,
                    level_n_components=fn_list)

                _wandb_log_viz(log_dict, coarse_model, real_coarse_eval, coarse_level_dims, epoch+1,
                               False, gen=gen_coarse_cpu, level_names=coarse_level_names)

                if decoder is not None and pca_models is not None:
                    rolls = decode_flow_to_piano_rolls(
                        gen_coarse_cpu, gen_fine_cpu,
                        pca_models, coarse_level_dims, fine_level_dims,
                        fine_levels_idx, cfg, decoder, device, n_samples=16)
                    grid = make_grid(rolls[:16], nrow=4, normalize=True)
                    log_dict['media/piano_rolls_gen'] = wandb.Image(grid, caption=f'Epoch {epoch+1}')

                _wandb_log_jacobian(log_dict, eval_model, real_fine_eval[:256],
                                    cond=real_coarse_eval[:256].to(device), device=device)
                gc.collect()

            fine_model.train()

        if (wandb.run is not None): wandb.log(log_dict, step=global_step)

        save_checkpoint((ema_model, fine_model), epoch+1, avg_loss, cfg or {},
                         optimizer=optimizer, save_every=save_every, tag=cfg.tag)

    if (wandb.run is not None): wandb.finish()


In [ ]:
#| export
#| eval: false
import hydra
from omegaconf import DictConfig

def _run_flow2(cfg: DictConfig):
    """Second-stage conditional fine-level flow training (called from train_flow_main)."""
    from midi_rae.data import ConditionalFlowDataset
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    print(f"device = {device}")

    flow2 = cfg.flow2
    fine_levels = list(flow2.get('fine_levels', [4, 5]))

    fine_pca_dir      = flow2.get('fine_pca_dir', None)
    fine_n_components = flow2.get('fine_n_components', None)
    if fine_pca_dir:
        fine_pca_dir      = os.path.expandvars(os.path.expanduser(str(fine_pca_dir)))
        fine_n_components = (list(fine_n_components) if hasattr(fine_n_components, '__iter__') else int(fine_n_components)) if fine_n_components is not None else None

    dataset = ConditionalFlowDataset(
        pca_dir           = flow2.pca_dir,
        encoded_dir       = flow2.get('encoded_dir', None),
        pca_levels        = list(flow2.get('pca_levels',  ['L0','L1','L2','L3'])),
        fine_levels       = fine_levels,
        emb_key           = flow2.get('emb_key', 'emb1'),
        fine_pca_dir      = fine_pca_dir,
        fine_n_components = fine_n_components,
    )
    print(f"  {len(dataset)} samples  "
          f"coarse_dims={dataset.coarse_level_dims}  fine_dims={dataset.fine_level_dims}")

    coarse_level_dims = dataset.coarse_level_dims
    coarse_model = CrossLevelFlowModel(
        level_dims   = coarse_level_dims,
        h_dim        = cfg.flow.h_dim,
        n_layers     = cfg.flow.n_layers,
        n_attn_layers= cfg.flow.get('n_attn_layers', 2),
        n_heads      = cfg.flow.get('n_heads', 8),
        t_dim        = cfg.flow.get('t_dim', 64),
    )
    from midi_rae.utils import load_checkpoint
    coarse_ckpt = flow2.get('coarse_ckpt', None)
    if coarse_ckpt and str(coarse_ckpt).lower() not in ('none', 'null', ''):
        coarse_ckpt = os.path.expandvars(os.path.expanduser(str(coarse_ckpt)))
        coarse_model = load_checkpoint(coarse_model, coarse_ckpt)
        print(f"  Loaded coarse model from {coarse_ckpt}")
    else:
        print("  coarse_ckpt not set — coarse model starts from random weights")

    # Determine fine_n_components for ConditionalFineFlowModel
    # In unified mode (no fine_pca_dir), derive from training.pca_n_per_lvl or dataset dims
    if fine_n_components is None:
        raw_npl = cfg.training.get('pca_n_per_lvl', None)
        if raw_npl is not None:
            fine_n_components = [list(raw_npl)[li] for li in fine_levels]
        else:
            print("  WARNING: fine_n_components not set; ConditionalFineFlowModel may fail")

    fine_model = ConditionalFineFlowModel(
        cond_dims     = coarse_level_dims,
        target_dims   = dataset.fine_level_dims,
        target_n_comp = fine_n_components,
        h_dim         = flow2.h_dim,
        n_layers      = flow2.n_layers,
        t_dim         = cfg.flow.get('t_dim', 64),
    )
    n_params = sum(p.numel() for p in fine_model.parameters())
    print(f"  ConditionalFineFlowModel: {n_params:,} parameters")

    # Load decoder + PCA models for piano roll visualization (optional)
    decoder, pca_models = None, None
    decoder_ckpt = os.path.expandvars(os.path.expanduser(str(cfg.generate.get('decoder_ckpt', '') or '')))
    if decoder_ckpt:
        import pickle
        from pathlib import Path
        from midi_rae.swin import SwinDecoder
        from midi_rae.train_dec import load_pca_models
        m = cfg.model
        decoder = SwinDecoder(
            img_height=cfg.data.image_size, img_width=cfg.data.image_size,
            patch_h=m.patch_h, patch_w=m.patch_w, out_channels=cfg.data.in_channels,
            embed_dim=m.embed_dim, depths=list(m.dec_depths),
            num_heads=list(m.dec_num_heads), window_size=m.window_size,
            mlp_ratio=m.mlp_ratio, drop_path_rate=0.0)
        decoder = load_checkpoint(decoder, decoder_ckpt).to(device).eval()
        for p in decoder.parameters(): p.requires_grad_(False)
        raw_npl = cfg.training.get('pca_n_per_lvl', None)
        n_per_lvl = list(raw_npl) if raw_npl is not None else None
        pca_dir = Path(os.path.expandvars(os.path.expanduser(flow2.pca_dir)))
        n_all_levels = len(coarse_level_dims) + len(fine_levels)
        pca_models = load_pca_models(str(pca_dir), n_all_levels, n_per_lvl=n_per_lvl)
        print(f"  Loaded decoder + {len(pca_models)} PCA models for piano roll viz")

    train_flow_conditional(coarse_model, fine_model, dataset, cfg,
                           device=device, decoder=decoder, pca_models=pca_models)


@hydra.main(version_base=None, config_path="../configs", config_name="config_swin")
def train_flow_main(cfg: DictConfig):
    tag = cfg.get('tag', '???')
    mode = cfg.get('flow_mode', 'flow2')
    stage = int(cfg.get('flow_stage', 1))
    if mode == 'flow2' or stage == 2:
        _run_flow2(cfg)
    else:
        _run_flow(cfg)

if __name__ == '__main__':
    train_flow_main()


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()